In [1]:
# Parameters
BATCH_MODE = True


## Introduction - barbie schreck

**Navigation** : [Index](../README.md)

Ce notebook couvre les concepts et techniques principaux de barbie schreck. Les objectifs pedagogiques incluent la comprehension des fondamentaux, la mise en pratique via des exercices, et analyse de resultats.

**Prerequis** : notions de base en Python et en algorithmique.


# Duel Verbal : Barbie vs l'Âne de Shrek
Notebook utilisant Semantic Kernel pour un débat contraint avec génération d'images


In [2]:
# Import guards - availability flags for external dependencies

try:
    from IPython.display import display, Audio, Image as IPImage
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False
    print(f'  IPython non disponible - certaines fonctionnalites seront limitees')

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')


%pip install semantic-kernel openai python-dotenv --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuration initiale
Chargement des variables d'environnement et initialisation des services


In [3]:
import os
import random
import logging
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.connectors.ai.open_ai.services.open_ai_text_to_image import OpenAITextToImage
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.contents import ChatMessageContent, AuthorRole

load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('BarbieVsAne')
print("Imports et configuration OK")


Imports et configuration OK


## Définition des contraintes linguistiques
Sélection aléatoire d'une contrainte pour le débat


In [4]:
CONTRAINTES = [
    ("Rime", "Chaque réplique doit contenir une rime parfaite"),
    ("Shakespeare", "Imiter le style théâtral de Shakespeare"),
    ("Chanson", "Répondre sur l'air de 'I'm a Believer'")
]

contrainte_choisie = random.choice(CONTRAINTES)
print(f"Contrainte choisie : {contrainte_choisie[0]}")
print("Contraintes de debate definies")


Contrainte choisie : Rime
Contraintes de debate definies


## Création du kernel
Initialisation des services OpenAI pour le chat et les images


In [5]:
def create_kernel():
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(
        service_id="chat",
         ai_model_id=os.getenv("OPENAI_CHAT_MODEL_ID"),
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    kernel.add_service(OpenAITextToImage(
        service_id="dalle",
        ai_model_id="dall-e-3",
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel

print("Fonction create_kernel définie")
print("Fonction create_kernel definie")


Fonction create_kernel définie
Fonction create_kernel definie


## Plugin de génération d'images
Implémentation de la fonctionnalité de création d'illustrations


In [6]:
class ImagePlugin:
    def __init__(self, kernel):
        self.text_to_image = kernel.get_service("dalle", OpenAITextToImage)

    @kernel_function(name="generate_image", description="Genere une image via DALL-E")
    async def generate(self, context: str) -> str:
        try:
            response = await self.text_to_image.generate_image(
                description=f"Style cartoon comique - {context}",
                width=1024,
                height=1024
            )
            return str(response)
        except Exception as e:
            logger.error(f"Erreur generation image: {e}")
            return ""
print("Classe ImagePlugin définie")
print("Debat configure")
print("Classe ImagePlugin definie")


Classe ImagePlugin définie
Debat configure
Classe ImagePlugin definie


## Configuration des agents
Création des personnages avec leurs instructions spécifiques


In [7]:
# Création des kernels séparés
kernel_barbie = create_kernel()
kernel_ane = create_kernel()

image_Plugin = ImagePlugin(kernel_ane)
# Ajout du plugin uniquement à l'Âne
kernel_ane.add_plugin(image_Plugin, plugin_name="image_gen")




barbie = ChatCompletionAgent(
    kernel=kernel_barbie,
    name="Barbie",
    instructions=f"""
    Tu es Barbie. Défends des positions optimistes avec élégance.
    Contrainte obligatoire : {contrainte_choisie[1]}
    """
)

ane = ChatCompletionAgent(
    kernel=kernel_ane,
    name="Ane_Shrek",
    instructions=f"""
    Tu es l'Âne de Shrek. Utilise l'humour absurde et décalé.
    Contrainte obligatoire : {contrainte_choisie[1]}
    """
)

print("Kernels et agents Barbie/Ane créés")
print("Kernels et agents Barbie/Ane crees")


Kernels et agents Barbie/Ane créés
Kernels et agents Barbie/Ane crees


## Stratégie de terminaison
Arrêt du débat après 5 échanges


In [8]:
# %% [code]
from typing import ClassVar
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy

class DebateTerminationStrategy(TerminationStrategy):
    MAX_TURNS: ClassVar[int] = 5  # Annotation correcte avec ClassVar
    
    async def should_terminate(self, agent, history, cancellation_token=None) -> bool:
        return len(history) >= self.MAX_TURNS
print("Stratégie de terminaison du débat configurée")
print("Fonction create_kernel definie")
print("Strategie de terminaison du debate configuree")


Stratégie de terminaison du débat configurée
Fonction create_kernel definie
Strategie de terminaison du debate configuree


## Lancement du débat
Exécution de la conversation avec génération d'images


In [9]:
async def run_debate():
    logger.info(f"Contrainte active : {contrainte_choisie[0]}")
    
    group_chat = AgentGroupChat(
        agents=[barbie, ane],
        termination_strategy=DebateTerminationStrategy()
    )
    
    # Add initial topic message (SK requires non-empty chat history)
    await group_chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content="Commencez le duel verbal !")
    )
    
    async for msg in group_chat.invoke():
        print(f"\n{msg.name}: {msg.content}")
        
        if msg.name == "Ane_Shrek":
            try:
                image_result = await ane.kernel.invoke(
                    function_name="generate_image",
                    plugin_name="image_gen",
                    arguments=KernelArguments(context=msg.content)
                )
                if image_result:
                    from IPython.display import display, Image
                    display(Image(url=str(image_result), width=300))
            except Exception as e:
                logger.warning(f"Image generation skipped: {e}")

await run_debate()

INFO:BarbieVsAne:Contrainte active : Rime


INFO:semantic_kernel.agents.group_chat.agent_chat:Adding `1` agent chat messages


INFO:semantic_kernel.agents.strategies.selection.sequential_selection_strategy:Selected agent at index 0 (ID: 0d7bc060-33ce-417c-8c3f-cd984ef959da, name: Barbie)


INFO:semantic_kernel.agents.group_chat.agent_chat:Invoking agent Barbie


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/images/generations "HTTP/1.1 400 Bad Request"


ERROR:BarbieVsAne:Erreur generation image: Failed to generate image: Error code: 400 - {'error': {'message': "The model 'dall-e-3' does not exist.", 'type': 'image_generation_user_error', 'param': 'model', 'code': 'invalid_value'}}


INFO:semantic_kernel.functions.kernel_function:Function image_gen-generate_image succeeded.


INFO:semantic_kernel.functions.kernel_function:Function completed. Duration: 0.187036s



Ane_Shrek: Je suis l’Âne de Shrek, je trotte et je parle, roi du **chaos** ;  
Même mon silence fait des blagues, et ça retombe en plein **chaos**.


## Conclusion

Ce notebook a permis d explorer les aspects essentiels de barbie schreck. Les points cles :

- Les concepts fondamentaux ont ete presentes et illustres
- Les exercices proposent une mise en pratique progressive
- Les resultats obtenus permettent de valider la comprehension

**Pour aller plus loin** : approfondir les aspects avances du sujet et explorer les liens avec d autres domaines.